# Analysis: reproduction (Step 1) and preliminary noise extension (Step 2)

Loads only the CSVs in `results/` and writes all figures to `figures/`. Nothing is simulated here. See `REPORT.md` for the discussion and for what these results do **not** show.

In [1]:
import os, json, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
import common as C, noise_analysis as NA
os.makedirs(C.FIGURES, exist_ok=True)
COL = {"HM": "#1f5fa8", "random": "#c0612b"}
LAB = {"HM": "hierarchical modular (HM)", "random": "random k-out null"}
pd.set_option("display.width", 160)

## Step 1: MC and KR vs link reciprocity (binary, N = 64, k = 6)

In [2]:
d1 = pd.read_csv(os.path.join(C.RESULTS, "step1_mc_kr.csv"))
print(d1.groupby("network_type").instance.nunique().rename("n_instances").to_frame().T.to_string())
print("rho used:", d1.rho_used.unique(), "| links per network: min/max", d1.n_links.min(), d1.n_links.max(),
      "| max |achieved - target r|:", (d1.achieved_reciprocity - d1.reciprocity).abs().max().round(3))
g = d1.groupby(["network_type", "reciprocity"])[["MC", "KR"]].agg(["mean", "std"]).round(3)
display(g)

network_type  HM  random
n_instances   50     250
rho used: [0.95] | links per network: min/max 382 384 | max |achieved - target r|: 0.005


MC            KR       
                           mean    std   mean    std
network_type reciprocity                            
HM           0.0          8.767  0.339  6.160  0.370
             0.1          8.720  0.286  6.120  0.328
             0.2          8.694  0.327  6.100  0.303
             0.3          8.529  0.394  6.100  0.364
             0.4          8.486  0.360  5.920  0.340
             0.5          8.339  0.329  5.960  0.348
             0.6          8.257  0.392  5.860  0.405
             0.7          8.088  0.376  5.820  0.388
             0.8          8.031  0.383  5.760  0.431
             0.9          7.943  0.392  5.780  0.418
             1.0          7.853  0.378  5.660  0.479
random       0.0          8.331  0.257  5.936  0.261
             0.1          8.274  0.251  5.912  0.298
             0.2          8.179  0.246  5.908  0.290
             0.3          8.157  0.265  5.924  0.266
             0.4          8.078  0.271  5.836  0.371
             0.5          7.974  0.320  5.788  0.410
             0.6          7.917  0.314  5.720  0.467
             0.7          7.807  0.289  5.644  0.496
             0.8          7.739  0.312  5.572  0.512
             0.9          7.690  0.300  5.568  0.512
             1.0          7.529  0.340  5.388  0.504

In [3]:
fits = {}
for kind in ["HM", "random"]:
    x = d1[d1.network_type == kind]
    for m in ["MC", "KR"]:
        means = x.groupby("reciprocity")[m].mean()
        fits[(kind, m, "level means (n=11)")] = C.linfit(means.index.values, means.values)
        fits[(kind, m, "all instances")] = C.linfit(x.reciprocity.values, x[m].values)
ft = pd.DataFrame(fits).T[["slope", "r2", "p", "n"]]
display(ft)
ft.to_csv(os.path.join(C.RESULTS, "step1_fits_table.csv"))

slope        r2             p       n
HM     MC level means (n=11) -0.979572  0.987305  7.609769e-10    11.0
          all instances      -0.979572  0.426477  3.624852e-68   550.0
       KR level means (n=11) -0.500000  0.951258  3.290008e-07    11.0
          all instances      -0.500000  0.147256  9.730061e-21   550.0
random MC level means (n=11) -0.774865  0.988386  5.096946e-10    11.0
          all instances      -0.774865  0.416641  0.000000e+00  2750.0
       KR level means (n=11) -0.527273  0.919359  3.214663e-06    11.0
          all instances      -0.527273  0.139948  4.423203e-92  2750.0

In [4]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, m, fk in zip(axs, ["MC", "KR"], ["MC", "KR"]):
    for kind in ["HM", "random"]:
        s = d1[d1.network_type == kind].groupby("reciprocity")[m].agg(["mean", "std"])
        ax.errorbar(s.index, s["mean"], s["std"], fmt="o-", ms=4, capsize=2, color=COL[kind], label=LAB[kind])
    f = fits[("HM", m, "level means (n=11)")]
    xs = np.array([0, 1.0]); ax.plot(xs, f["intercept"] + f["slope"] * xs, "--", color=COL["HM"], alpha=.5, lw=1)
    ax.set_xlabel("link reciprocity r"); ax.set_ylabel({"MC": "memory capacity", "KR": "kernel rank"}[m])
    ax.set_title(f"HM fit: slope {f['slope']:+.2f}, R² {f['r2']:.3f}", fontsize=10)
axs[0].legend(fontsize=8)
fig.suptitle("Step 1 reproduction - mean ± SD over instances (binary, N=64, k=6)", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(C.FIGURES, "step1_mc_kr_vs_reciprocity.png"), dpi=160)
plt.close(fig)
print("saved figures/step1_mc_kr_vs_reciprocity.png")

saved figures/step1_mc_kr_vs_reciprocity.png


## Step 2 (PRELIMINARY): robustness to state noise

In [5]:
d2 = pd.read_csv(os.path.join(C.RESULTS, "step2_noise.csv"))
SIG = sorted(d2.sigma.unique())
print("networks per type:", d2.groupby("network_type").apply(lambda x: x[["instance","reciprocity"]].drop_duplicates().shape[0]).to_dict())
mc = d2.groupby(["network_type", "reciprocity", "sigma"]).MC.agg(["mean", "std"]).reset_index()
display(mc.pivot_table(index=["network_type", "reciprocity"], columns="sigma", values="mean").round(2))

networks per type: {'HM': 250, 'random': 1250}


sigma                     0.0000  0.0001  0.0003  0.0010  0.0030  0.0100  0.0300  0.1000
network_type reciprocity                                                                
HM           0.00           8.96    7.79    6.59    5.34    4.14    2.75    1.49    0.39
             0.25           8.82    7.60    6.38    5.15    3.99    2.65    1.41    0.35
             0.50           8.53    7.34    6.14    4.94    3.82    2.52    1.33    0.32
             0.75           8.31    7.09    5.91    4.77    3.70    2.43    1.29    0.32
             1.00           8.03    6.87    5.71    4.58    3.54    2.33    1.22    0.29
random       0.00           8.53    7.35    6.19    4.99    3.88    2.61    1.42    0.38
             0.25           8.39    7.22    6.07    4.90    3.80    2.55    1.38    0.37
             0.50           8.16    7.00    5.87    4.72    3.66    2.44    1.31    0.34
             0.75           7.96    6.81    5.70    4.59    3.56    2.36    1.26    0.32
             1.00           7.71    6.58    5.49    4.41    3.42    2.27    1.21    0.29

In [6]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
x0 = 3e-5  # position used to display sigma = 0 on the log axis
cmap = plt.cm.viridis(np.linspace(0, .9, 5))
for ax, kind in zip(axs, ["HM", "random"]):
    for c, r in zip(cmap, sorted(d2.reciprocity.unique())):
        s = mc[(mc.network_type == kind) & (mc.reciprocity == r)].sort_values("sigma")
        xs = np.where(s.sigma.values == 0, x0, s.sigma.values)
        ax.plot(xs, s["mean"], "o-", ms=3.5, color=c, label=f"r = {r:g}")
        ax.fill_between(xs, s["mean"] - s["std"], s["mean"] + s["std"], color=c, alpha=.12, lw=0)
    ax.set_xscale("log"); ax.set_title(LAB[kind], fontsize=10); ax.set_xlabel("state-noise std σ (train and test)")
    ticks = [x0, 1e-4, 1e-3, 1e-2, 1e-1]; ax.set_xticks(ticks); ax.set_xticklabels(["0", "1e-4", "1e-3", "1e-2", "1e-1"])
axs[0].set_ylabel("memory capacity (mean ± SD)"); axs[0].legend(fontsize=8, title="link reciprocity")
fig.suptitle("Step 2 (preliminary): MC vs state noise", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(C.FIGURES, "step2_mc_vs_noise_by_reciprocity.png"), dpi=160)
plt.close(fig); print("saved")

saved


In [7]:
beta = NA.compute_beta_table(d2, SIG)
bs = NA.beta_summary(beta)
display(bs.round(3))
fig, ax = plt.subplots(figsize=(5.5, 4))
for kind, off in [("HM", -0.01), ("random", 0.01)]:
    s = bs[bs.network_type == kind].sort_values("reciprocity")
    ax.errorbar(s.reciprocity + off, s.beta_mean, yerr=[s.beta_mean - s.ci_low, s.ci_high - s.beta_mean],
                fmt="o-", capsize=3, color=COL[kind], label=LAB[kind])
ax.set_xlabel("link reciprocity r"); ax.set_ylabel("β  (log σ*ₙ ~ β · log n)")
ax.set_title("Noise-threshold exponent, mean and 95% bootstrap CI\n(across networks; censored delays excluded)", fontsize=9)
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(os.path.join(C.FIGURES, "step2_beta_vs_reciprocity.png"), dpi=160); plt.close(fig); print("saved")

,network_type,reciprocity,n_networks,n_beta_defined,beta_mean,beta_sd,ci_low,ci_high,mean_delays_used,mean_left_censored,mean_right_censored
0,HM,0.00,50,50,-2.995,0.115,-3.026,-2.963,7.580,1.420,0.0
1,HM,0.25,50,50,-3.089,0.098,-3.116,-3.063,8.620,0.480,0.0
2,HM,0.50,50,50,-2.999,0.166,-3.042,-2.951,9.100,0.200,0.0
3,HM,0.75,50,50,-2.875,0.143,-2.915,-2.836,9.460,0.020,0.0
4,HM,1.00,50,50,-2.749,0.165,-2.795,-2.704,9.480,0.060,0.0
5,random,0.00,250,250,-3.064,0.105,-3.077,-3.051,7.252,1.572,0.0
6,random,0.25,250,250,-3.112,0.105,-3.125,-3.099,7.200,1.264,0.0
7,random,0.50,250,250,-3.164,0.098,-3.176,-3.152,8.056,0.548,0.0
8,random,0.75,250,250,-3.058,0.127,-3.073,-3.042,8.996,0.124,0.0
9,random,1.00,250,250,-2.925,0.151,-2.943,-2.905,9.044,0.108,0.0


saved


In [8]:
# per-delay noise tolerance (sigma*_n) for context: median over networks, by r (HM only)
r2cols = [c for c in d2.columns if c.startswith("R2_")]
delays = np.array([int(c[3:]) for c in r2cols])
fig, ax = plt.subplots(figsize=(5.5, 4))
for c, r in zip(cmap, sorted(d2.reciprocity.unique())):
    g0 = d2[(d2.network_type == "HM") & (d2.reciprocity == r)]
    mean_curve = g0.groupby("sigma")[r2cols].mean().loc[SIG].values
    ns, ss = [], []
    for i, n in enumerate(delays):
        if mean_curve[0, i] < NA.R2_FLOOR: continue
        s, st = NA.sigma_star(mean_curve[1:, i], SIG[1:], mean_curve[0, i])
        if st == "ok": ns.append(n); ss.append(s)
    ax.loglog(ns, ss, "o-", ms=3, color=c, label=f"r = {r:g}")
ax.set_xlabel("delay n"); ax.set_ylabel("σ*ₙ (instance-mean R²ₙ)"); ax.legend(fontsize=8)
ax.set_title("HM: noise level at which R²ₙ halves", fontsize=9); fig.tight_layout()
fig.savefig(os.path.join(C.FIGURES, "step2_sigma_star_vs_delay_HM.png"), dpi=160); plt.close(fig)

In [9]:
corr = NA.correlations(d2, beta, SIG)
show = corr[(corr.scope.isin(["HM", "all"]))]
display(show[show.target == "beta"].round(3))
display(show[(show.target == "MC") & (show.sigma.isin([1e-3, 1e-2, 3e-2]))].round(3))

,scope,target,sigma,predictor,n,spearman,p,partial_spearman_given_r,partial_p
16,HM,beta,NaN,henrici_index,250,-0.524,0.0,-0.027,0.673
17,HM,beta,NaN,spectral_gap,250,-0.318,0.0,-0.084,0.184
52,all,beta,NaN,henrici_index,1500,-0.325,0.0,-0.085,0.001
53,all,beta,NaN,spectral_gap,1500,-0.414,0.0,-0.288,0.000


,scope,target,sigma,predictor,n,spearman,p,partial_spearman_given_r,partial_p
6,HM,MC,0.001,henrici_index,250,0.679,0.000,-0.039,0.542
7,HM,MC,0.001,spectral_gap,250,0.216,0.001,-0.196,0.002
10,HM,MC,0.010,henrici_index,250,0.509,0.000,-0.052,0.411
11,HM,MC,0.010,spectral_gap,250,0.157,0.013,-0.133,0.036
12,HM,MC,0.030,henrici_index,250,0.429,0.000,-0.122,0.054
13,HM,MC,0.030,spectral_gap,250,0.169,0.007,-0.070,0.269
42,all,MC,0.001,henrici_index,1500,0.617,0.000,-0.030,0.240
43,all,MC,0.001,spectral_gap,1500,0.255,0.000,-0.341,0.000
46,all,MC,0.010,henrici_index,1500,0.481,0.000,-0.021,0.420
47,all,MC,0.010,spectral_gap,1500,0.230,0.000,-0.185,0.000


### Context tables for REPORT.md: noise retention, paired differences, sensitivity of beta, state amplitude

In [10]:
# (a) paired difference in MC between r=0 and r=1 (same HM instances) and fraction of noise-free MC retained
hm = d2[d2.network_type == "HM"]
p = hm.pivot_table(index=["instance", "sigma"], columns="reciprocity", values="MC").reset_index()
rows = []
for s, g_ in p.groupby("sigma"):
    diff = (g_[0.0] - g_[1.0]).values
    rows.append(dict(sigma=s, mean_MC_r0=g_[0.0].mean(), mean_MC_r1=g_[1.0].mean(), paired_diff=diff.mean(),
                     se=diff.std(ddof=1) / np.sqrt(len(diff)), ratio=g_[0.0].mean() / g_[1.0].mean()))
display(pd.DataFrame(rows).round(3))
ret = hm.pivot_table(index=["instance", "reciprocity"], columns="sigma", values="MC")
display((ret.div(ret[0.0], axis=0)).groupby("reciprocity").mean().round(3).rename_axis("MC(sigma)/MC(0), HM"))

,sigma,mean_MC_r0,mean_MC_r1,paired_diff,se,ratio
0,0.000,8.961,8.029,0.932,0.057,1.116
1,0.000,7.787,6.868,0.919,0.051,1.134
2,0.000,6.593,5.706,0.887,0.044,1.155
3,0.001,5.335,4.582,0.753,0.040,1.164
4,0.003,4.141,3.541,0.600,0.040,1.170
5,0.010,2.751,2.328,0.423,0.035,1.182
6,0.030,1.493,1.225,0.268,0.024,1.219
7,0.100,0.394,0.286,0.108,0.013,1.379


sigma,0.0000,0.0001,0.0003,0.0010,0.0030,0.0100,0.0300,0.1000
"MC(sigma)/MC(0), HM",,,,,,,,
0.00,1.0,0.869,0.736,0.595,0.462,0.307,0.167,0.044
0.25,1.0,0.861,0.723,0.583,0.453,0.301,0.160,0.040
0.50,1.0,0.861,0.720,0.579,0.448,0.296,0.156,0.037
0.75,1.0,0.853,0.712,0.575,0.445,0.293,0.156,0.038
1.00,1.0,0.855,0.711,0.571,0.441,0.290,0.152,0.035


In [11]:
# (b) sensitivity: beta fitted on a common delay window n <= 6 (removes r-dependent censoring of longer delays)
sens = NA.beta_summary(NA.compute_beta_table(d2, SIG, max_delay=6))
display(sens[["network_type", "reciprocity", "beta_mean", "ci_low", "ci_high", "mean_delays_used"]].round(3))
amp = pd.read_csv(os.path.join(C.RESULTS, "step2_state_amplitude.csv"))
display(amp.groupby(["network_type", "reciprocity"]).state_rms.mean().unstack().round(4).rename_axis("noise-free state RMS"))

,network_type,reciprocity,beta_mean,ci_low,ci_high,mean_delays_used
0,HM,0.00,-2.550,-2.576,-2.525,6.0
1,HM,0.25,-2.710,-2.742,-2.680,6.0
2,HM,0.50,-2.851,-2.888,-2.814,6.0
3,HM,0.75,-2.937,-2.972,-2.904,6.0
4,HM,1.00,-2.962,-2.999,-2.922,6.0
5,random,0.00,-2.717,-2.731,-2.703,6.0
6,random,0.25,-2.793,-2.808,-2.777,6.0
7,random,0.50,-2.911,-2.928,-2.895,6.0
8,random,0.75,-2.998,-3.012,-2.984,6.0
9,random,1.00,-3.039,-3.053,-3.023,6.0


reciprocity,0.00,0.25,0.50,0.75,1.00
noise-free state RMS,,,,,
HM,0.0341,0.0337,0.0336,0.0328,0.0327
random,0.0322,0.0320,0.0318,0.0322,0.0316


In [12]:
# save all derived tables (same as step2_noise.py writes, recomputed here from the CSV)
beta.to_csv(os.path.join(C.RESULTS, "step2_beta_per_network.csv"), index=False)
bs.to_csv(os.path.join(C.RESULTS, "step2_beta_summary.csv"), index=False)
corr.to_csv(os.path.join(C.RESULTS, "step2_correlations.csv"), index=False)
print(sorted(os.listdir(C.FIGURES)))

['step1_mc_kr_vs_reciprocity.png', 'step2_beta_vs_reciprocity.png', 'step2_mc_vs_noise_by_reciprocity.png', 'step2_sigma_star_vs_delay_HM.png']
